# 4장 1강: 데이터 불균형과 평가지표 왜곡

## 실습 목표

이번 실습에서는 **Credit Card Fraud** 데이터를 이용하여 데이터 불균형이 분류 모델 평가에 어떤 영향을 주는지 확인합니다.

- `Class` 비율을 확인하여 불균형 정도를 파악합니다.
- Accuracy만 보면 생길 수 있는 착시를 확인합니다.
- Precision, Recall, F1-score, PR-AUC를 함께 확인합니다.
- `class_weight`, Random Undersampling, SMOTE를 적용하여 성능 변화를 비교합니다.
- threshold 변화에 따른 Precision / Recall 변화를 확인합니다.

> 이 실습에서는 4장 1강 교안에 나온 내용만 사용합니다.

## 실습 환경 / 데이터

- Python
- pandas
- scikit-learn
- imbalanced-learn
- 데이터: `creditcard.csv`

### 주요 컬럼

| 컬럼 | 의미 |
|---|---|
| `V1` ~ `V28` | 거래 정보를 변환한 숫자형 특성 |
| `Class` | 정상 거래 0 / 사기 거래 1 |

이번 실습에서는 모델 입력으로 `V1`부터 `V28`까지 사용하고, `Class`를 예측 대상으로 사용합니다.

## 실습 준비

1. 데이터 크기를 확인합니다.
2. `Class`의 개수와 비율을 확인합니다.
3. `V1`~`V28`을 X, `Class`를 y로 분리합니다.
4. Train / Test 데이터를 분리합니다.
5. 리샘플링은 **Train 데이터에만** 적용합니다.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix
)

from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE

data = pd.read_csv("creditcard.csv")

print("데이터 크기:", data.shape)

print("\n[Class 개수]")
display(data["Class"].value_counts().to_frame("count"))

print("\n[Class 비율]")
display(data["Class"].value_counts(normalize=True).to_frame("ratio"))

features = [f"V{i}" for i in range(1, 29)]

X = data[features]
y = data["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTrain 크기:", X_train.shape)
print("Test 크기:", X_test.shape)

---

# 필수 1. Accuracy의 함정과 불균형 지표 확인

## 배경

Credit Card Fraud 데이터는 정상 거래가 매우 많고 사기 거래가 매우 적은 불균형 데이터입니다.

이런 상황에서는 Accuracy만 보면 모델이 매우 좋아 보일 수 있습니다.

## 문제 1. 모든 거래를 정상이라고 예측해보세요.

### 요구사항

1. Test 데이터의 모든 거래를 `0`으로 예측합니다.
2. Accuracy를 계산합니다.
3. Recall을 계산합니다.
4. Accuracy와 Recall 결과를 비교합니다.

### 확인 포인트

- 모든 거래를 정상이라고 예측해도 Accuracy가 매우 높게 나오는가?
- 사기 거래를 하나도 잡지 못하면 Recall은 어떻게 되는가?

In [ ]:
# TODO
# 모든 거래를 정상(0)으로 예측하고 Accuracy / Recall을 계산하세요.

### Q1. Accuracy가 높더라도 이 모델을 좋은 사기 탐지 모델이라고 보기 어려운 이유는 무엇인가요?

**답변:**  
여기에 작성하세요.

## 문제 2. 기본 Logistic Regression의 여러 지표를 확인하세요.

### 요구사항

1. `LogisticRegression(max_iter=1000)` 모델을 학습합니다.
2. Test 데이터를 예측합니다.
3. Accuracy를 계산합니다.
4. Precision을 계산합니다.
5. Recall을 계산합니다.
6. F1-score를 계산합니다.
7. `predict_proba()`의 양성 클래스 확률로 PR-AUC를 계산합니다.
8. 혼동행렬을 확인합니다.

### 확인 포인트

- Accuracy 하나만 보는 것과 여러 지표를 함께 보는 것의 차이를 이해했는가?
- Precision과 Recall이 각각 어떤 오류와 연결되는지 설명할 수 있는가?
- PR-AUC가 불균형 데이터에서 왜 중요한지 이해했는가?

In [ ]:
# TODO
# 기본 Logistic Regression의 Accuracy / Precision / Recall / F1 / PR-AUC / 혼동행렬을 확인하세요.

### Q2. 사기 거래를 놓치는 FN의 비용이 크다면 Precision과 Recall 중 무엇을 더 중요하게 봐야 하나요?

**답변:**  
여기에 작성하세요.

---

# 필수 2. 불균형 처리 기법 비교

## 배경

교안에서는 불균형 처리 방법으로 다음 세 가지를 다룹니다.

- `class_weight`
- Random Undersampling
- SMOTE

이번 문제에서는 세 방법을 Train 데이터에만 적용한 뒤 같은 Test 데이터에서 성능을 비교합니다.

## 문제 1. class_weight를 적용하세요.

### 요구사항

1. `LogisticRegression(class_weight="balanced", max_iter=1000)`을 사용합니다.
2. Train 데이터로 학습합니다.
3. Test 데이터에서 Precision, Recall, F1-score, PR-AUC를 계산합니다.

### 확인 포인트

- 소수 클래스에 더 큰 가중치를 주면 Recall이 어떻게 변하는가?
- Precision과 Recall이 동시에 항상 좋아지는 것은 아니라는 점을 확인했는가?

In [ ]:
# TODO
# class_weight="balanced" 모델을 학습하고 지표를 확인하세요.

## 문제 2. Random Undersampling을 적용하세요.

### 요구사항

1. `RandomUnderSampler(random_state=42)`를 사용합니다.
2. **Train 데이터에만** `fit_resample()`을 적용합니다.
3. 리샘플링 전후 클래스 개수를 확인합니다.
4. Logistic Regression을 학습합니다.
5. Test 데이터에서 Precision, Recall, F1-score, PR-AUC를 계산합니다.

### 확인 포인트

- 다수 클래스 샘플이 줄어드는가?
- Test 데이터에는 Undersampling을 적용하지 않았는가?

In [ ]:
# TODO
# Random Undersampling을 Train 데이터에만 적용하세요.

## 문제 3. SMOTE를 적용하세요.

### 요구사항

1. `SMOTE(random_state=42)`를 사용합니다.
2. **Train 데이터에만** `fit_resample()`을 적용합니다.
3. 리샘플링 전후 클래스 개수를 확인합니다.
4. Logistic Regression을 학습합니다.
5. Test 데이터에서 Precision, Recall, F1-score, PR-AUC를 계산합니다.

### 확인 포인트

- 소수 클래스 샘플이 증가하는가?
- SMOTE가 Test 데이터에 적용되지 않았는가?
- Recall 개선과 Precision 감소가 함께 나타날 수 있음을 확인했는가?

In [ ]:
# TODO
# SMOTE를 Train 데이터에만 적용하세요.

### Q3. 리샘플링을 Test 데이터에도 적용하면 안 되는 이유는 무엇인가요?

**답변:**  
여기에 작성하세요.

---

# 과제 1. 불균형 처리 전후 성능과 threshold 비교

## 배경

필수 실습에서는 기본 모델과 세 가지 불균형 처리 방법을 확인했습니다.

이번 과제에서는 **기본 Logistic Regression과 class_weight 모델**을 비교하고, class_weight 모델에서 threshold에 따른 Precision / Recall 변화를 확인합니다.

> 과제는 필수 실습에서 사용한 지표와 방법을 그대로 비교하는 수준입니다.

## 요구사항

1. 기본 Logistic Regression과 `class_weight="balanced"` 모델의 다음 지표를 하나의 DataFrame으로 정리합니다.
   - Accuracy
   - Precision
   - Recall
   - F1-score
   - PR-AUC
2. 두 모델의 Recall과 Precision 차이를 설명합니다.
3. `class_weight="balanced"` 모델의 양성 확률을 사용합니다.
4. threshold를 `0.3`, `0.5`, `0.7`로 바꾸어 예측합니다.
5. 각 threshold에서 Precision과 Recall을 계산합니다.
6. threshold 변화에 따라 두 지표가 어떻게 달라지는지 설명합니다.
7. 사기 거래를 놓치는 FN의 비용이 크다고 가정할 때, threshold를 낮추는 방향이 어떤 의미인지 설명합니다.

8. 정상 거래만 예측해도 Accuracy가 높게 나오는 이유와 이 지표만 사용할 때의 문제를 설명합니다.
9. FN의 비용과 데이터 특성을 고려해 `class_weight`, Random Undersampling, SMOTE 중 우선 적용할 방법을 하나 선택하고 이유와 한계를 설명합니다. 추가 모델 구현은 요구하지 않습니다.
10. Q4~Q7에 모두 답합니다.

> 처리 기법 선택은 근거 있는 제안으로 작성합니다. Test 결과를 보고 처리 기법이나 threshold를 반복 조정하지 않으며, 실제 설정 선택에는 Train 내부 검증 또는 교차검증이 필요합니다.

### 확인 포인트

- 불균형 처리 전후를 Accuracy 하나가 아니라 여러 지표로 비교했는가?
- threshold를 낮추면 Recall이 높아지는 경향을 설명할 수 있는가?
- threshold를 높이면 Precision이 높아지는 경향을 설명할 수 있는가?
- 단일한 정답 threshold가 있는 것이 아니라 비용 구조에 따라 결정해야 한다는 점을 이해했는가?
- 클래스 비율 때문에 Accuracy가 사기 탐지 성능을 오해하게 할 수 있음을 설명했는가?
- FN 비용과 데이터 특성을 근거로 처리 기법의 선택 이유·한계를 제시했는가?


In [ ]:
# TODO
# 기본 모델 / class_weight 모델 비교와 threshold 0.3, 0.5, 0.7 비교를 진행하세요.

### Q4. 사기 거래를 놓치는 비용이 매우 크다면 threshold를 낮추는 방향은 어떤 장점과 단점이 있나요?

**답변:**  
여기에 작성하세요.

### Q5. CTGAN / TVAE와 SMOTE의 가장 큰 차이를 한 문장으로 설명하세요.

**답변:**  
여기에 작성하세요.

### Q6. 정상 거래만 예측해도 Accuracy가 높게 나오는 이유와 이 지표만으로 사기 탐지 성능을 판단할 때의 문제를 설명하세요.

**답변:**
여기에 작성하세요.


### Q7. 사기 거래를 놓치는 FN의 비용과 데이터 특성을 고려해 class_weight·Random Undersampling·SMOTE 중 우선 적용할 방법을 하나 선택하고, 선택 이유와 한계를 설명하세요.

**답변:**
여기에 작성하세요.


---

# 실습 마무리

아래 질문에 짧게 답해보세요.

1. 불균형 데이터에서 Accuracy 하나만 보면 위험한 이유는 무엇인가요?
2. Precision은 어떤 오류가 중요할 때 보는 지표인가요?
3. Recall은 어떤 오류가 중요할 때 보는 지표인가요?
4. class_weight, Undersampling, SMOTE는 각각 어떤 방식으로 불균형을 다루나요?
5. 리샘플링은 Train / Test 중 어디에만 적용해야 하나요?
6. threshold를 낮추면 일반적으로 Recall과 Precision은 어떻게 변하나요?